In [1]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

## 1. Imports and Model Loading

In [2]:

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
import uuid
import imageio
import numpy as np
from IPython.display import Image as ImageDisplay
import sys
sys.path.append("Deep_Learning/course_proj/sam-3d-objects/notebook")
from inference import Inference, ready_gaussian_for_video_rendering, load_image, load_masks, display_image, make_scene, render_video, interactive_visualizer

/scratch/chenguo/.conda/envs/sam3d/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2026-01-03 18:40:05.298 | INFO     | sam3d_objects.pipeline.inference_pipeline:set_attention_backend:15 - GPU name is NVIDIA A40
/scratch/chenguo/.conda/envs/sam3d/lib/python3.11/site-packages/lightning/fabric/__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
2026-01-03 18:40:07.067 | INFO     | sam3d_objects.model.backbone.tdfy_dit.modules.sparse:__from_env:39 - [SPARSE] Backend: spconv, Attention: sdpa
/scratch/chenguo/.conda/envs/sam3d/lib/python3.11/site-p

[SPARSE][CONV] spconv algo: auto


2026-01-03 18:40:12.332 | WARNING  | sam3d_objects.data.dataset.tdfy.preprocessor:__post_init__:51 - No rgb pointmap normalizer provided, using scale + shift 
2026-01-03 18:40:12.333 | WARNING  | sam3d_objects.data.dataset.tdfy.preprocessor:__post_init__:51 - No rgb pointmap normalizer provided, using scale + shift 


In [3]:
PATH = os.getcwd()
print(PATH)
TAG = "hf"
# config_path = f"{PATH}/../checkpoints/{TAG}/pipeline.yaml"
config_path = f"{PATH}/Deep_Learning/course_proj/sam-3d-objects/checkpoints/{TAG}/checkpoints/pipeline.yaml"
inference = Inference(config_path, compile=False)

/scratch/chenguo/course/Deep_Learning/course_proj/sam-3d-objects/notebook


FileNotFoundError: [Errno 2] No such file or directory: '/scratch/chenguo/course/Deep_Learning/course_proj/sam-3d-objects/notebook/Deep_Learning/course_proj/sam-3d-objects/checkpoints/hf/checkpoints/pipeline.yaml'

## 2. Load input image to lift to 3D (multiple objects)

In [ ]:
IMAGE_PATH = f"{PATH}/Deep_Learning/course_proj/sam-3d-objects/notebook/images/shutterstock_stylish_kidsroom_1640806567/image.png"
IMAGE_NAME = os.path.basename(os.path.dirname(IMAGE_PATH))

image = load_image(IMAGE_PATH)
masks = load_masks(os.path.dirname(IMAGE_PATH), extension=".png")
display_image(image, masks)

## 3. Generate Gaussian Splats

In [ ]:
outputs = [inference(image, mask, seed=42) for mask in masks]

## 4. Visualize Gaussian Splat of the Scene
### a. Animated Gif

In [ ]:
scene_gs = make_scene(*outputs)
scene_gs = ready_gaussian_for_video_rendering(scene_gs)
output_dir = f"{PATH}/Deep_Learning/course_proj/sam-3d-objects/gaussians/multi"
os.makedirs(output_dir, exist_ok=True)
# export gaussian splatting (as point cloud)
scene_gs.save_ply(f"{output_dir}/{IMAGE_NAME}.ply")

video = render_video(
    scene_gs,
    r=1,
    fov=60,
    resolution=512,
)["color"]

# save video as gif
imageio.mimsave(
    os.path.join(f"{PATH}/Deep_Learning/course_proj/sam-3d-objects/gaussians/multi/{IMAGE_NAME}.gif"),
    video,
    format="GIF",
    duration=1000 / 30,  # default assuming 30fps from the input MP4
    loop=0,  # 0 means loop indefinitely
)

# notebook display
ImageDisplay(url=f"gaussians/multi/{IMAGE_NAME}.gif?cache_invalidator={uuid.uuid4()}",)

### b. Interactive Visualizer

In [ ]:
# might take a while to load (black screen)
interactive_visualizer(f"{PATH}/Deep_Learning/course_proj/sam-3d-objects/gaussians/multi/{IMAGE_NAME}.ply")